# Sub-Doppler rubidium atom cooling using a programmable agile integrated PZT-on-SiN resonator

**Andrei Isichenko**<sup>1†</sup>, **Steven Carpenter**<sup>2†</sup>, **Nick Montifiore**<sup>1</sup>, **Jiawei Wang**<sup>1</sup>, **Mayand Dangi**<sup>3</sup>, **Nitesh Chauhan**<sup>1</sup>, **Pritha Mukherjee**<sup>3</sup>, **Xuting Yang**<sup>4</sup>, **Nitin Indukuri**<sup>1</sup>, **Mark W. Harrington**<sup>1</sup>, **Chuan Zhong**<sup>1</sup>, **Iain M. Kierzewski**<sup>5</sup>, **Ryan Q. Rudy**<sup>5</sup>, **Jennifer T. Choy**<sup>2,3∗</sup>, and **Daniel J. Blumenthal**<sup>1∗</sup>

<sup>1</sup>Department of Electrical and Computer Engineering, University of California Santa Barbara, Santa Barbara, California 93106, USA

<sup>2</sup>Department of Physics, University of Wisconsin-Madison, Madison, Wisconsin 53706, USA

<sup>3</sup>Department of Electrical and Computer Engineering, University of Wisconsin-Madison, Madison, Wisconsin 53706, USA

<sup>4</sup>Department of Materials Science and Engineering, University of Wisconsin-Madison, Madison, Wisconsin 53706, USA

<sup>5</sup>U.S. Army Research Laboratory, Adelphi, Maryland 20783, USA

†These authors contributed equally: Andrei Isichenko, Steven Carpenter

∗Corresponding Authors: jennifer.choy@wisc.edu, danb@ucsb.edu

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline

# Local NCOMMS data repository (run kernel with this folder as cwd)
DATA_REPO = Path.cwd()
assert (DATA_REPO / "resonators.py").exists(), (
    f"Expected resonators.py in cwd; got {DATA_REPO}. "
    "Open/run the notebook from the data_repository folder."
)
plt.style.use(DATA_REPO / "plot_style.mplstyle")

from resonators import (
    calibrate_mzi,
    crop_rescale_res_mzi,
    linear_fit,
    moving_average,
    q_fit_lz,
)

def data_join(*parts):
    """Absolute path under DATA_REPO."""
    return str(DATA_REPO.joinpath(*parts))

def data_list_files(*parts, suffix=None):
    """Basenames in a DATA_REPO subdirectory (optional suffix filter, e.g. '.tiff')."""
    d = DATA_REPO.joinpath(*parts)
    names = [p.name for p in d.iterdir() if p.is_file() and not p.name.startswith(".")]
    if suffix is not None:
        names = [n for n in names if n.endswith(suffix)]
    return names


## Figure 2

### Fig. 2b

In [ ]:
def normalize(vec):
    return vec/np.max(vec)


base_path = data_join('figure2', 'fig2b')
filename = 'Die6_1ring_1bus_d6_1p15um_gap_MZIFSR_24MHz_1.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.4 # MHz

wg_ng = 1.671032 # from tidy3d: https://tidy3d.simulation.cloud/workbench?taskId=pa-353017eb-834b-4e7a-809e-ceadcda93109

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 10, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 50, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [10, 100], fit_model= 0, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=False)

plt.figure(figsize=(6,4))
plt.plot(q_data['f detuning'], q_data['y res'], color='red', lw=2, label='Resonance')
fsr_mzi = q_data['MZIFSR']
plt.plot(q_data['f detuning'], q_data['y mzi'], color='#2177b4',label=f'{fsr_mzi:.1f} MHz MZI')
plt.plot(q_data['f detuning'], q_data['y fit'],  c='black', ls='--', lw=2, label='Fit')
plt.xlim((min(q_data['f detuning'])+100, max(q_data['f detuning'])-90))
plt.ylim((0, 1.1))
plt.xlabel('Detuning (MHz)')
plt.ylabel('Transmission (a.u.)')
plt.legend(fontsize=14)
plt.tight_layout()
plt.show()


### Fig. 2c

In [ ]:
def get_pna_data(path):
    df = pd.read_csv(path, header=5, encoding='unicode_escape')
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df.dropna(how='all')
    return df

plt.figure(figsize=(8, 6))
plt.subplot(2,1,1)

base_path = data_join('figure2', 'fig2c')
filename = f'die6_d6_S21_BPD_5.csv'
path = os.path.join(base_path, filename)
df_die6_d6 = get_pna_data(path)
S21_amp_die6_d6_run = df_die6_d6['S21(DB)'] + 3.9 # - np.mean(df_die6_d6['S21(DB)'][0:10])

plt.subplot(2,1,1)
plt.semilogx(df_die6_d6['Freq(Hz)'], S21_amp_die6_d6_run, lw=2)
plt.axhline(-3, ls='--', color='k', lw=1, label='-3 dB')
plt.axhline(-6, ls='--', color='r', lw=1, label='-6 dB')
plt.axvline(4.5e6, ls='--', color='k')
plt.axvline(11e6, ls='--', color='red')
plt.legend()
plt.ylabel('S$_{21}$ (dB)')
plt.ylim([-15, 3])
plt.xlim([1e4, 3e7])
plt.title('die 6')
plt.subplot(2,1,2)
plt.semilogx(df_die6_d6['Freq(Hz)'], df_die6_d6['S21(DEG)'], lw=2)
plt.axhline(180, ls='--', lw=1, color='k')
plt.axhline(0, ls='--', lw=2, color='k', label=r'$f_{180^{\circ}}$')
plt.axvline(5.8e6, ls='--', color='k')
plt.ylabel('Phase (deg)') 
plt.xlabel('Frequency (Hz)')
plt.xlim([1e4, 3e7])
plt.legend()
#plt.savefig('PNA_die6_AC_meas.pdf')
plt.show()


### Fig. 2d

In [ ]:
debug = False
include_split = False
fit_model = 0
lw_guess = [100, 100.0]

smooth_val = 25

MZI_FSR = 24.8 # MHz

PZT_voltages = [0.0,0.5,1.0,1.5,2.0,3.0, 3.5,4.0,4.5,5.0,5.5,6.0,6.5,7.0,7.5,8.0,8.5,9.0,9.5,10.0]

res_freq_vals = []

base_path = data_join('figure2', 'fig2de')

plt.figure(figsize=(6,5))

Die_num = 6
device_num = 6 # gap 1.15 um

for voltage in PZT_voltages:
    voltage_str = str(voltage).replace('.', 'p')
    filename = f'Q_780PZT_die6_singlering_device6_MZIFSR24p8MHz_35p228C_PZT_{voltage_str}V.csv'
    #print(filename)
    path = os.path.join(base_path, filename)
        
    df = pd.read_csv(path)
    start_idx = 0
    end_idx = -1
    df = df[start_idx:end_idx]
    
    Nmean = 1
    y_res = df['trans_pd'].to_numpy()
    y_res = moving_average(y_res, Nmean)  # moving average
    y_res = y_res/np.max(y_res) # normalized data

    y_mzi = df['mzi_pd'].to_numpy()
    y_mzi = moving_average(y_mzi, Nmean)  # moving average
    y_mzi = y_mzi / np.max(y_mzi)*0.2  # normalized data

    freq = calibrate_mzi(y_mzi, fsr_mzi = MZI_FSR, Nmean = 2, Nbetween = 5, peaks_debug = debug) 

    q_data = q_fit_lz(freq, y_res, y_mzi, fsr_mzi=MZI_FSR, lw_guess = lw_guess, 
                      wl_res = 780e-9, fit_model=fit_model, add_drop=False, wg_ng = 1.48, fit_plot=False, print_result=False)

    freq = freq - freq[0]
    
    res_resample = q_data['y res']
    mzi_resample = q_data['y mzi']
    y_fit = q_data['y fit']
    
    peak_position = freq[np.argmin(y_fit)]
    res_freq_vals.append(peak_position)
    
    if voltage==0:
        plt.plot(freq/1e3, moving_average(q_data['y mzi'],1) * 1 + 0.1, '-g', linewidth=1, label='MZI')
    
    transmission_to_plot = normalize(res_resample)
    plt.plot(freq/1e3, transmission_to_plot, linewidth=2)#, label=f'PZT = {voltage:.1f} V')
    plt.scatter(peak_position/1e3, transmission_to_plot[np.argmin(res_resample)], marker='x')

plt.xlabel('Detuning (GHz)', fontsize=20)
plt.ylabel('Transmission (a.u.)', fontsize=20)
plt.ylim([0,1.1])
plt.xlim([2, 14])
plt.show()

plt.figure(figsize=(6,3))
plt.plot(freq/1e3, moving_average(q_data['y mzi'],1) * 1 + 0.1, '-g', linewidth=0.5, label='MZI')
plt.xlabel('Detuning (GHz)')
plt.xlim([0,0.5])
plt.show()


### Fig. 2e

In [ ]:
PZT_voltages = np.asarray(PZT_voltages)

res_shift = res_freq_vals - res_freq_vals[0]
m, b = linear_fit(PZT_voltages, res_shift)
plt.figure(figsize=(5,5))
plt.plot(PZT_voltages, (res_freq_vals - res_freq_vals[0])/1e3,'o', label='data')
plt.plot(PZT_voltages, (m*PZT_voltages + b)/1e3,'--', lw=2, color='k', label=f'fit, {m:.0f} MHz/V')
plt.xlabel('Applied DC bias (V)', fontsize=20)
plt.ylabel('Resonance shift (GHz)', fontsize=20)
plt.legend(fontsize=16)
plt.show()


### Fig. 2f

Add-thru resonators: non-split Qmeasure and DC tuning (dies 2, 3, 4, 7, 8, 10).


In [ ]:
FIG2F_DATA = DATA_REPO / 'figure2' / 'fig2f'

def fig2f_join(*parts):
    """Absolute path under the fig2f data directory."""
    return str(FIG2F_DATA.joinpath(*parts))

def fig2f_csv_names(*parts):
    """CSV basenames in a fig2f subdirectory."""
    d = FIG2F_DATA.joinpath(*parts)
    return [p.name for p in d.glob('*.csv')]


#### Die 2

##### Q-measure

In [ ]:
base_path = fig2f_join('die2')
filename = 'Qmeasure_Die2_1ring_1bus_1p15um_gap_MZIFSR_24p8MHz_780p15nm_run1.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.8 # MHz

wg_ng = 1.4837

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 3, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 10, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [100, 100], fit_model= 0, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=True)

base_path = fig2f_join('die2')
filename = 'Qmeasure_Die2_1ring_1bus_1p15um_gap_MZIFSR_24p8MHz_780p23nm_run1.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.8 # MHz

wg_ng = 1.4837

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 3, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 10, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [100, 100], fit_model= 0, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=True)



##### DC tuning

In [ ]:

debug = False
include_split = False

smooth_val = 25

MZI_FSR = 24.8 # MHz

base_path = fig2f_join('die2', 'DC_tune_add_thru_run1')
filenames = fig2f_csv_names('die2', 'DC_tune_add_thru_run1')

# Sort filenames by the voltage at the end of the string
filenames.sort(key=lambda x: float(x.split('_')[-1][:-5].replace('p', '.')))

res_freq_vals = []
PZT_voltages = []

fig, axs = plt.subplots(1, 2, figsize=(12, 5))  # Create side by side subplots
fig.suptitle('die 2, one bus, g=1.15 um', fontsize=16)

for filename in filenames:
    voltage = float(filename.split('_')[-1][:-5].replace('p', '.'))
    PZT_voltages.append(voltage)
    path = os.path.join(base_path, filename)
        
    df = pd.read_csv(path)
    start_idx = 0
    end_idx = -1
    df = df[start_idx:end_idx]
    
    Nmean = 1
    y_res = df['trans_pd'].to_numpy()
    y_res = moving_average(y_res, Nmean)  # moving average
    y_res = y_res/np.max(y_res) # normalized data

    y_mzi = df['mzi_pd'].to_numpy()
    y_mzi = moving_average(y_mzi, Nmean)  # moving average
    y_mzi = y_mzi / np.max(y_mzi)*0.2  # normalized data

    freq = calibrate_mzi(y_mzi, fsr_mzi = MZI_FSR, Nmean = 3, Nbetween = 10, peaks_debug = debug) 

    q_data = q_fit_lz(freq, y_res, y_mzi, fsr_mzi=MZI_FSR, lw_guess = [10,10], 
                      wl_res = 780e-9, fit_model=0, add_drop=False, wg_ng = 1.48, fit_plot=False, print_result=False)

    freq = freq - freq[0]
    
    res_resample = q_data['y res']
    mzi_resample = q_data['y mzi']
    y_fit = q_data['y fit']
    
    peak_position = freq[np.argmin(y_fit)]
    res_freq_vals.append(peak_position)
    
    if voltage==0:
        axs[0].plot(freq/1e3, moving_average(q_data['y mzi'],1) * 1 + 0.1, '-g', linewidth=1, label='MZI')
    
    transmission_to_plot = normalize(res_resample)
    axs[0].plot(freq/1e3, transmission_to_plot, linewidth=2)
    axs[0].scatter(peak_position/1e3, transmission_to_plot[np.argmin(res_resample)], marker='x')

axs[0].set_xlabel('Detuning (GHz)', fontsize=20)
axs[0].set_ylabel('Transmission (a.u.)', fontsize=20)
axs[0].set_ylim([-0.1,1.1])
axs[0].set_xlim([0, 15])


PZT_voltages = np.asarray(PZT_voltages)

res_shift = res_freq_vals - res_freq_vals[0]
m, b = linear_fit(PZT_voltages, res_shift)
axs[1].plot(PZT_voltages, (res_freq_vals - res_freq_vals[0])/1e3,'o', label='data')
axs[1].plot(PZT_voltages, (m*PZT_voltages + b)/1e3,'--', lw=2, color='k', label=f'fit, {m:.0f} MHz/V')
axs[1].set_xlabel('Applied DC bias (V)', fontsize=20)
axs[1].set_ylabel('Resonance shift (GHz)', fontsize=20)
axs[1].legend(fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to make room for the title
plt.show()


#### Die 3

##### Q-measure

In [ ]:
base_path = fig2f_join('die3')
filename = 'Qmeasure_Die3_1ring_1bus_d6_1p15um_gap_MZIFSR_24p8MHz_780p16nm_run1.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.8 # MHz

wg_ng = 1.4837

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 3, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 10, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [100, 100], fit_model= 0, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=True)


##### DC tuning

In [ ]:
debug = False
include_split = False

smooth_val = 25

MZI_FSR = 24.8 # MHz

base_path = fig2f_join('die3', 'DC_tune_run2')
filenames = fig2f_csv_names('die3', 'DC_tune_run2')

# Sort filenames by the voltage at the end of the string
filenames.sort(key=lambda x: float(x.split('_')[-1][:-5].replace('p', '.')))

res_freq_vals = []
PZT_voltages = []

fig, axs = plt.subplots(1, 2, figsize=(12, 5))  # Create side by side subplots
fig.suptitle('die 3, d6, one bus, g=1.15 um', fontsize=16)

for filename in filenames:
    voltage = float(filename.split('_')[-1][:-5].replace('p', '.'))
    PZT_voltages.append(voltage)
    path = os.path.join(base_path, filename)
        
    df = pd.read_csv(path)
    start_idx = 0
    end_idx = -1
    df = df[start_idx:end_idx]
    
    Nmean = 1
    y_res = df['trans_pd'].to_numpy()
    y_res = moving_average(y_res, Nmean)  # moving average
    y_res = y_res/np.max(y_res) # normalized data

    y_mzi = df['mzi_pd'].to_numpy()
    y_mzi = moving_average(y_mzi, Nmean)  # moving average
    y_mzi = y_mzi / np.max(y_mzi)*0.2  # normalized data

    freq = calibrate_mzi(y_mzi, fsr_mzi = MZI_FSR, Nmean = 3, Nbetween = 10, peaks_debug = debug) 

    q_data = q_fit_lz(freq, y_res, y_mzi, fsr_mzi=MZI_FSR, lw_guess = [10,10], 
                      wl_res = 780e-9, fit_model=0, add_drop=False, wg_ng = 1.48, fit_plot=False, print_result=False)

    freq = freq - freq[0]
    
    res_resample = q_data['y res']
    mzi_resample = q_data['y mzi']
    y_fit = q_data['y fit']
    
    peak_position = freq[np.argmin(y_fit)]
    res_freq_vals.append(peak_position)
    
    if voltage==0:
        axs[0].plot(freq/1e3, moving_average(q_data['y mzi'],1) * 1 + 0.1, '-g', linewidth=1, label='MZI')
    
    transmission_to_plot = normalize(res_resample)
    axs[0].plot(freq/1e3, transmission_to_plot, linewidth=2)
    axs[0].scatter(peak_position/1e3, transmission_to_plot[np.argmin(res_resample)], marker='x')

axs[0].set_xlabel('Detuning (GHz)', fontsize=20)
axs[0].set_ylabel('Transmission (a.u.)', fontsize=20)
axs[0].set_ylim([0,1.1])
axs[0].set_xlim([6, 11])


PZT_voltages = np.asarray(PZT_voltages)

res_shift = res_freq_vals - res_freq_vals[0]
m, b = linear_fit(PZT_voltages, res_shift)
axs[1].plot(PZT_voltages, (res_freq_vals - res_freq_vals[0])/1e3,'o', label='data')
axs[1].plot(PZT_voltages, (m*PZT_voltages + b)/1e3,'--', lw=2, color='k', label=f'fit, {m:.0f} MHz/V')
axs[1].set_xlabel('Applied DC bias (V)', fontsize=20)
axs[1].set_ylabel('Resonance shift (GHz)', fontsize=20)
axs[1].legend(fontsize=16)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to make room for the title
plt.show()


#### Die 4

##### Q-measure

In [ ]:
base_path = fig2f_join('die4')
filename = 'Qmeasure_Die4_1ring_1bus_d6_1p15um_gap_MZIFSR_24p8MHz_run2.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.8 # MHz

wg_ng = 1.4837

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 3, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 10, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [10, 100], fit_model= 0, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=True)


##### DC tuning

In [ ]:
debug = False
include_split = False

smooth_val = 25

MZI_FSR = 24.8 # MHz

base_path = fig2f_join('die4', 'DC_tune_run1')
filenames = fig2f_csv_names('die4', 'DC_tune_run1')

# Sort filenames by the voltage at the end of the string
filenames.sort(key=lambda x: float(x.split('_')[-1][:-5].replace('p', '.')))
filenames
plt.figure(figsize=(6,5))

res_freq_vals = []
PZT_voltages = []
for filename in filenames: 
    voltage = float(filename.split('_')[-1][:-5].replace('p', '.'))
    PZT_voltages.append(voltage)
    path = os.path.join(base_path, filename)
        
    df = pd.read_csv(path)
    start_idx = 100
    end_idx = -300
    df = df[start_idx:end_idx]
    
    Nmean = 1
    y_res = df['trans_pd'].to_numpy()
    y_res = moving_average(y_res, Nmean)  # moving average
    y_res = y_res/np.max(y_res) # normalized data
    
    y_mzi = df['mzi_pd'].to_numpy()
    y_mzi = moving_average(y_mzi, Nmean)  # moving average
    y_mzi = y_mzi / np.max(y_mzi)*0.2  # normalized data
    freq = calibrate_mzi(y_mzi, fsr_mzi = MZI_FSR, Nmean = 4, Nbetween = 6, peaks_debug = False) 

    q_data = q_fit_lz(freq, y_res, y_mzi, fsr_mzi=MZI_FSR, lw_guess = [10,10], 
                    wl_res = 780e-9, fit_model=0, add_drop=False, wg_ng = 1.48, fit_plot=False, print_result=False)

    freq = freq - freq[0]
    
    res_resample = q_data['y res']
    mzi_resample = q_data['y mzi']
    y_fit = q_data['y fit']
    
    peak_position = freq[np.argmin(y_fit)]
    res_freq_vals.append(peak_position)
        
    if voltage==0:
        plt.plot(freq/1e3, moving_average(q_data['y mzi'],1) * 1 + 0.1, '-g', linewidth=1, label='MZI')
    
    transmission_to_plot = normalize(res_resample)
    plt.plot(freq/1e3, transmission_to_plot, linewidth=2)#, label=f'PZT = {voltage:.1f} V')
    plt.scatter(peak_position/1e3, transmission_to_plot[np.argmin(res_resample)], marker='x')

plt.xlabel('Detuning (GHz)', fontsize=20)
plt.ylabel('Transmission (a.u.)', fontsize=20)
plt.ylim([0,1.1])
plt.xlim([2, 12])
plt.show()


PZT_voltages = np.asarray(PZT_voltages)

res_shift = res_freq_vals - res_freq_vals[0]
m, b = linear_fit(PZT_voltages, res_shift)
plt.figure(figsize=(5,5))
plt.plot(PZT_voltages, (res_freq_vals - res_freq_vals[0])/1e3,'o', label='data')
plt.plot(PZT_voltages, (m*PZT_voltages + b)/1e3,'--', lw=2, color='k', label=f'fit, {m:.0f} MHz/V')
plt.xlabel('Applied DC bias (V)', fontsize=20)
plt.ylabel('Resonance shift (GHz)', fontsize=20)
plt.legend(fontsize=16)
plt.show()


#### Die 7

##### Q-measure

In [ ]:
base_path = fig2f_join('die7')
filename = 'Die7_1ring_1bus_d6_1p15um_gap_MZIFSR_24MHz_PZT_0p0V.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.8 # MHz

wg_ng = 1.4837

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 3, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 10, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [100, 100,10], fit_model= 1, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=True)


##### DC tuning

In [ ]:
debug = False
include_split = False

smooth_val = 25

MZI_FSR = 24 # MHz

base_path = fig2f_join('die7')
filenames = fig2f_csv_names('die7')

# Sort filenames by the voltage at the end of the string
filenames.sort(key=lambda x: float(x.split('_')[-1][:-5].replace('p', '.')))

res_freq_vals = []
PZT_voltages = []

fig, axs = plt.subplots(1, 2, figsize=(12, 5))  # Create side by side subplots
fig.suptitle('die 7, one bus, g=1.15 um', fontsize=16)

for filename in filenames:
    voltage = float(filename.split('_')[-1][:-5].replace('p', '.'))
    PZT_voltages.append(voltage)
    path = os.path.join(base_path, filename)
        
    df = pd.read_csv(path)
    start_idx = 100
    end_idx = -100
    df = df[start_idx:end_idx]
    
    Nmean = 20
    y_res = df['trans_pd'].to_numpy()
    y_res = moving_average(y_res, Nmean)  # moving average
    y_res = y_res/np.max(y_res) # normalized data

    y_mzi = df['mzi_pd'].to_numpy()
    y_mzi = moving_average(y_mzi, Nmean)  # moving average
    y_mzi = y_mzi / np.max(y_mzi)*0.2  # normalized data

    freq = calibrate_mzi(y_mzi, fsr_mzi = MZI_FSR, Nmean = 15, Nbetween = 50, peaks_debug = debug)

    q_data = q_fit_lz(freq, y_res, y_mzi, fsr_mzi=MZI_FSR, lw_guess = [100, 100.0], 
                        wl_res = 780e-9, fit_model=0, add_drop=False, wg_ng = 1.48, fit_plot=False, print_result=False)

    freq = freq - freq[0]
    
    res_resample = q_data['y res']
    mzi_resample = q_data['y mzi']
    y_fit = q_data['y fit']
    
    peak_position = freq[np.argmin(y_fit)]
    res_freq_vals.append(peak_position)
    
    if voltage==0:
        axs[0].plot(freq/1e3, moving_average(q_data['y mzi'],1) * 1 + 0.1, '-g', linewidth=1, label='MZI')
    
    transmission_to_plot = normalize(res_resample)
    axs[0].plot(freq/1e3, transmission_to_plot, linewidth=2)
    axs[0].scatter(peak_position/1e3, transmission_to_plot[np.argmin(res_resample)], marker='x')

axs[0].set_xlabel('Detuning (GHz)', fontsize=20)
axs[0].set_ylabel('Transmission (a.u.)', fontsize=20)
axs[0].set_ylim([-0.1,1.1])
axs[0].set_xlim([0, 2.5])


PZT_voltages = np.asarray(PZT_voltages)
res_shift = res_freq_vals - res_freq_vals[0]
m, b = linear_fit(PZT_voltages, res_shift)
axs[1].plot(PZT_voltages, (res_freq_vals - res_freq_vals[0])/1e3,'o', label='data')
axs[1].plot(PZT_voltages, (m*PZT_voltages + b)/1e3,'--', lw=2, color='k', label=f'fit, {m:.0f} MHz/V')
axs[1].set_xlabel('Applied DC bias (V)', fontsize=20)
axs[1].set_ylabel('Resonance shift (GHz)', fontsize=20)
axs[1].legend(fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to make room for the title
plt.show()


#### Die 8

##### Q-measure

In [ ]:
base_path = fig2f_join('die8')
filename = 'Qmeasure_Die8_1ring_1bus_1p15um_gap_MZIFSR_24p8MHz_780p24nm_non_split_res.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.8 # MHz

wg_ng = 1.4837

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 3, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 10, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [100, 100], fit_model= 0, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=True)


##### DC tuning

In [ ]:
debug = False
include_split = False

smooth_val = 25

MZI_FSR = 24.8 # MHz

base_path = fig2f_join('die8', 'DC_tune_add_thru_run1')
filenames = fig2f_csv_names('die8', 'DC_tune_add_thru_run1')

# Sort filenames by the voltage at the end of the string
filenames.sort(key=lambda x: float(x.split('_')[-1][:-5].replace('p', '.')))

res_freq_vals = []
PZT_voltages = []

fig, axs = plt.subplots(1, 2, figsize=(12, 5))  # Create side by side subplots
fig.suptitle('die 8, one bus, g=1.15 um', fontsize=16)

for filename in filenames:
    voltage = float(filename.split('_')[-1][:-5].replace('p', '.'))
    PZT_voltages.append(voltage)
    path = os.path.join(base_path, filename)
        
    df = pd.read_csv(path)
    start_idx = 0
    end_idx = -1
    df = df[start_idx:end_idx]
    
    Nmean = 1
    y_res = df['trans_pd'].to_numpy()
    y_res = moving_average(y_res, Nmean)  # moving average
    y_res = y_res/np.max(y_res) # normalized data

    y_mzi = df['mzi_pd'].to_numpy()
    y_mzi = moving_average(y_mzi, Nmean)  # moving average
    y_mzi = y_mzi / np.max(y_mzi)*0.2  # normalized data

    freq = calibrate_mzi(y_mzi, fsr_mzi = MZI_FSR, Nmean = 3, Nbetween = 10, peaks_debug = debug) 

    q_data = q_fit_lz(freq, y_res, y_mzi, fsr_mzi=MZI_FSR, lw_guess = [10,10], 
                      wl_res = 780e-9, fit_model=0, add_drop=True, wg_ng = 1.48, fit_plot=False, print_result=False)

    freq = freq - freq[0]
    
    res_resample = q_data['y res']
    mzi_resample = q_data['y mzi']
    y_fit = q_data['y fit']
    
    peak_position = freq[np.argmin(y_fit)]
    res_freq_vals.append(peak_position)
    
    if voltage==0:
        axs[0].plot(freq/1e3, moving_average(q_data['y mzi'],1) * 1 + 0.1, '-g', linewidth=1, label='MZI')
    
    transmission_to_plot = normalize(res_resample)
    axs[0].plot(freq/1e3, transmission_to_plot, linewidth=2)
    axs[0].scatter(peak_position/1e3, transmission_to_plot[np.argmin(res_resample)], marker='x')

axs[0].set_xlabel('Detuning (GHz)', fontsize=20)
axs[0].set_ylabel('Transmission (a.u.)', fontsize=20)
axs[0].set_ylim([-0.1,1.1])
axs[0].set_xlim([0, 15])


PZT_voltages = np.asarray(PZT_voltages)

res_shift = res_freq_vals - res_freq_vals[0]
m, b = linear_fit(PZT_voltages, res_shift)
axs[1].plot(PZT_voltages, (res_freq_vals - res_freq_vals[0])/1e3,'o', label='data')
axs[1].plot(PZT_voltages, (m*PZT_voltages + b)/1e3,'--', lw=2, color='k', label=f'fit, {m:.0f} MHz/V')
axs[1].set_xlabel('Applied DC bias (V)', fontsize=20)
axs[1].set_ylabel('Resonance shift (GHz)', fontsize=20)
axs[1].legend(fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to make room for the title
plt.show()


#### Die 10

##### Q-measure

In [ ]:
base_path = fig2f_join('die10')
filename = 'Qmeasure_Die10_1ring_1bus_1p15um_gap_MZIFSR_24p8MHz_780p16nm_non_split_res.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.8 # MHz

wg_ng = 1.4837

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 3, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 10, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [100, 100], fit_model= 0, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=True)




In [ ]:
base_path = fig2f_join('die10')
filename = 'Qmeasure_Die10_1ring_1bus_1p15um_gap_MZIFSR_24p8MHz_780p18nm_non_split_res.csv'
path = os.path.join(base_path, filename)

mzi_fsr = 24.8 # MHz

wg_ng = 1.4837

y_res, y_mzi = crop_rescale_res_mzi(path, X_window = 0, Nmean = 3, mzi_debug = False)
f = calibrate_mzi(y_mzi, fsr_mzi = mzi_fsr, Nmean = 15, Nbetween = 10, peaks_debug = True)
q_data = q_fit_lz(f, y_res, y_mzi, fsr_mzi=mzi_fsr, lw_guess = [100, 100], fit_model= 0, add_drop=False, wl_res = 780e-9, wg_ng = wg_ng, fit_plot=True)


##### DC tuning

In [ ]:
debug = False
include_split = False

smooth_val = 25

MZI_FSR = 24.8 # MHz

base_path = fig2f_join('die10', 'DC_tune_add_thru_run1')
filenames = fig2f_csv_names('die10', 'DC_tune_add_thru_run1')

# Sort filenames by the voltage at the end of the string
filenames.sort(key=lambda x: float(x.split('_')[-1][:-5].replace('p', '.')))

res_freq_vals = []
PZT_voltages = []

fig, axs = plt.subplots(1, 2, figsize=(12, 5))  # Create side by side subplots
fig.suptitle('die 10, one bus, g=1.15 um', fontsize=16)

for filename in filenames:
    voltage = float(filename.split('_')[-1][:-5].replace('p', '.'))
    PZT_voltages.append(voltage)
    path = os.path.join(base_path, filename)
        
    df = pd.read_csv(path)
    start_idx = 0
    end_idx = -1
    df = df[start_idx:end_idx]
    
    Nmean = 3
    y_res = df['trans_pd'].to_numpy()
    y_res = moving_average(y_res, Nmean)  # moving average
    y_res = y_res/np.max(y_res) # normalized data

    y_mzi = df['mzi_pd'].to_numpy()
    y_mzi = moving_average(y_mzi, Nmean)  # moving average
    y_mzi = y_mzi / np.max(y_mzi)*0.2  # normalized data

    freq = calibrate_mzi(y_mzi, fsr_mzi = MZI_FSR, Nmean = 3, Nbetween = 10, peaks_debug = debug) 

    q_data = q_fit_lz(freq, y_res, y_mzi, fsr_mzi=MZI_FSR, lw_guess = [100,100], 
                      wl_res = 780e-9, fit_model=0, add_drop=False, wg_ng = 1.48, fit_plot=False, print_result=False)

    freq = freq - freq[0]
    
    res_resample = q_data['y res']
    mzi_resample = q_data['y mzi']
    y_fit = q_data['y fit']
    
    peak_position = freq[np.argmin(y_fit)]
    res_freq_vals.append(peak_position)
    
    if voltage==0:
        axs[0].plot(freq/1e3, moving_average(q_data['y mzi'],1) * 1 + 0.1, '-g', linewidth=1, label='MZI')
    
    transmission_to_plot = normalize(res_resample)
    axs[0].plot(freq/1e3, transmission_to_plot, linewidth=2)
    axs[0].scatter(peak_position/1e3, transmission_to_plot[np.argmin(res_resample)], marker='x')

axs[0].set_xlabel('Detuning (GHz)', fontsize=20)
axs[0].set_ylabel('Transmission (a.u.)', fontsize=20)
axs[0].set_ylim([-0.1,1.1])
axs[0].set_xlim([0, 15])


PZT_voltages = np.asarray(PZT_voltages)

res_shift = res_freq_vals - res_freq_vals[0]
m, b = linear_fit(PZT_voltages, res_shift)
axs[1].plot(PZT_voltages, (res_freq_vals - res_freq_vals[0])/1e3,'o', label='data')
axs[1].plot(PZT_voltages, (m*PZT_voltages + b)/1e3,'--', lw=2, color='k', label=f'fit, {m:.0f} MHz/V')
axs[1].set_xlabel('Applied DC bias (V)', fontsize=20)
axs[1].set_ylabel('Resonance shift (GHz)', fontsize=20)
axs[1].legend(fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to make room for the title
plt.show()


## Figure 4

In [ ]:
import tifffile as tiff
from scipy.ndimage import uniform_filter
import math

# %% region Temperature fitting
def sigmacalc(hwhm):
    sigma = hwhm / np.sqrt(2*math.log(2))
    return sigma

def HWHM(sigma):
    r = np.sqrt(2*math.log(2))*sigma
    return r

import re
def extract_time_ms(filename):
    # This regex will extract the number just before 'ms.tiff'
    match = re.search(r'(\d+)_0ms\.tiff$', filename)
    if match:
        return int(match.group(1))
    else:
        return float('inf')  # Place any unmatched at the end

def import_photos(filenames, base_path, movie_name='TOF_movie.tiff'):
    # Sort filenames by time in ms
    # Extract number before '_0ms.tiff' and convert to integer for proper numerical sorting
    filenames = sorted(filenames, key=extract_time_ms)

    # Load list of TIFF file paths
    image_files = [os.path.join(base_path, f) for f in filenames]

    # Read all images into a numpy array
    images = np.stack([tiff.imread(img) for img in image_files], axis=0)

    return images

def import_flight_times(filenames):
    # Sort filenames by time in ms
    sorted_files = sorted(filenames, key=extract_time_ms)
    flight_times = np.array([extract_time_ms(f) for f in sorted_files])

    return flight_times

def run_gauss_fit(img, axis='y'):
    """ Used for the sum-fitting (instead of fit along a slice of image. """
    if axis=='y':
        sumaxis = 0
        pix_len = img.shape[1]
    if axis=='z':
        sumaxis = 1
        pix_len = img.shape[0]

    ## integrate along axis
    from scipy.optimize import curve_fit

    pix = np.linspace(0, pix_len, pix_len)
    signal = np.sum(img,axis=sumaxis)
    signal = signal - np.min(signal)
    amp = max(signal)
    amp_idx = np.argmax(signal)

    def gaussian(x, a, x0, sigma, c):
        return a * np.exp(-(x - x0)**2 / (2 * sigma**2)) + c

    # Guess initial parameters: (height, center, width, offset)
    initial_guess = [max(signal), pix[np.argmax(signal)], 1, min(signal)]
    # Fit!
    popt, pcov = curve_fit(gaussian, pix, signal, p0=initial_guess)
    # popt contains best-fit parameters: [a, x0, sigma]
    # print(f"Best-fit parameters: a = {popt[0]:.2f}, x0 = {popt[1]:.2f}, sigma = {popt[2]:.2f}")
    sigma = popt[2]
    Gauss = gaussian(pix, popt[0], popt[1], popt[2], popt[3])

    return pix, signal, Gauss, sigma, amp, amp_idx

def extract_temperature_init_sigma(tof_ms, sigma_pix, axis='y', magnification=1):
    """
    Assume the sigma pix std error is 1
    """
    from scipy.optimize import curve_fit
    M_Rb87 = 1.443160648e-25 # kg, from Steck Rb87
    from scipy.constants import Boltzmann as kB  # units J/K

    mm_per_um = 1/1e3
    sigma_pix_mm = sigma_pix * mm_per_um *magnification

    def TempFit(x, a):
        f = sigma_pix_mm[0]**2 + a * x
        return f

    # assuming M = 0.9
    # 1 pix std * mm_per_pix**2
    sig2_error = 4.24e-5

    t2 = tof_ms**2
    r2 = (sigma_pix_mm)**2
    # print("Radius square is: ", r2)
    fitT, covT= curve_fit(TempFit, t2, r2)#, sigma=np.ones(len(t2))*sig2_error)
    m = fitT[0]
    mm2_per_ms2_to_m2_per_s2 = (1e-3 / 1e-3)**2

    Temp = 1e6 * (m * M_Rb87 / kB) # units: uK
    fitted = m * t2 + sigma_pix_mm[0]**2
    Terr = 1e6 * (np.sqrt(np.diag(covT)[0]) * M_Rb87 / kB) # units: uK

    return r2, t2, Temp, Terr, fitted

def plot_t2_r2(tof_vals_final, sigma_y_vals_final, sigma_z_vals_final):

    magnification_tof = 1
    ry2, t2, Tempy, Terr_y, fittedy = extract_temperature_init_sigma(tof_vals_final, sigma_y_vals_final, axis='y', magnification=magnification_tof)
    rz2, t2, Tempz, Terr_z, fittedz = extract_temperature_init_sigma(tof_vals_final, sigma_z_vals_final, axis='z', magnification=magnification_tof)

    figure = plt.figure(figsize=(6,4))
    plt.plot(t2,ry2,'o', color='red', label=rf'$\sigma_y$: Temp={Tempy:.0f} $\pm$ {Terr_y:.0f} uK')
    plt.plot(t2,fittedy, color='red', ls='--')

    plt.plot(t2,rz2,'o', color='blue', label=rf'$\sigma_z$: Temp={Tempz:.0f} $\pm$ {Terr_z:.0f} uK')
    plt.plot(t2,fittedz, color='blue', ls='--')

    plt.ylabel(rf'$\sigma^2$ (mm$^2$)', fontsize=16)
    plt.xlabel('$t^2$ (ms$^2$)', fontsize=16)
    plt.legend()
    plt.title("Time-of-flight measurement")
    plt.tight_layout()
    plt.show()
    return {
        "figure": figure,
        "ry2": ry2,
        "rz2": rz2,
        "fittedy": fittedy,
        "fittedz": fittedz,
        "Tempy": Tempy,
        "Terr_y": Terr_y,
        "Tempz": Tempz,
        "Terr_z": Terr_z
    }

def plot_calc_temp_full(base_path, filenames, plotting=False, save_result = True):

    # Sort files numerically:
    tiff_files = sorted([f for f in filenames],
                    key=lambda x: int(x.split('_')[0]))

    # import the data (Steven's way):
    arr = import_photos([s for s in tiff_files], base_path)
    flight_times = import_flight_times(tiff_files)

    microns_per_pixel = 12.86 # M = 1 for Steven's system

    ysigmavals = np.zeros(len(arr))
    zsigmavals = np.zeros(len(arr))
    ysize = np.zeros(len(arr))
    zsize = np.zeros(len(arr))

    for idx in range(len(arr)):
        raw_img = arr[idx,:,:]#arr[0,400:500,425:525]

        img_for_fit = uniform_filter(raw_img, 1)

        ypix, signaly, GaussY, sigmay, ampy, ampy_idx = run_gauss_fit(img_for_fit, axis='y')
        zpix, signalz, GaussZ, sigmaz, ampz, ampz_idx = run_gauss_fit(img_for_fit, axis='z')

        if plotting:
            fig = plt.figure(figsize=(12,3))
            plt.subplot(1,3,1)
            plt.imshow(img_for_fit, vmax = 1.2 * np.max(img_for_fit))
            clb = plt.colorbar()
            clb.ax.set_title('pix',fontsize=10)
            plt.subplot(1,3,2)

            plt.plot(ypix, signaly, marker='.', label='data, y')
            plt.plot(ypix, GaussY, label=rf'fit, $\sigma_y$ = {sigmay:.0f}')
            plt.legend()
            plt.xlabel('y [pixels]')
            plt.title(f'image sumy')
            plt.subplot(1,3,3)

            plt.plot(zpix, signalz, marker='.', label='data, z')
            plt.plot(zpix, GaussZ, label=rf'fit, $\sigma_z$ = {sigmaz:.0f}')
            plt.legend()
            plt.xlabel('z [pixels]')
            plt.title(f'image sumz')
            plt.suptitle(f'TOF = {flight_times[idx]:.1f} ms')
            plt.show()

        ysigmavals[idx] = sigmay*microns_per_pixel
        zsigmavals[idx] = sigmaz*microns_per_pixel

    print("Y sigmas: ", ysigmavals)
    print("Z sigmas: ", zsigmavals)

    ####################################

    tof_vals = flight_times #np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0])

    ry = HWHM(ysigmavals) #Andrei Method
    rz = HWHM(zsigmavals) #Andrei Method

    sigy = sigmacalc(ysize) #Steven Method
    sigz = sigmacalc(zsize) #Steven Method

    result_dict = plot_t2_r2(tof_vals, ysigmavals, zsigmavals)
    figure = result_dict["figure"]
    fittedy = result_dict["fittedy"]
    fittedz = result_dict["fittedz"]
    ry2 = result_dict["ry2"]
    rz2 = result_dict["rz2"]

    if save_result:
        figure.savefig(base_path+"results", dpi=300, bbox_inches='tight')  # dpi=300 is high-res

    # Include all elements from result_dict in output_dict, except for 'figure', along with tof_vals, ysigmavals, zsigmavals
    output_dict = {
        'tof_vals': tof_vals,
        "ry2": ry2,
        "rz2": rz2,
        'fittedy': result_dict['fittedy'],
        'fittedz': result_dict['fittedz'],
        "Tempy": result_dict['Tempy'],
        "Terr_y": result_dict['Terr_y'],
        "Tempz": result_dict['Tempz'],
        "Terr_z": result_dict['Terr_z']   
    }

    return output_dict


### I. No PGC

In [ ]:
base_base_path = data_join('figure4')
base_path = os.path.join(base_base_path, 'Trial_7_no_subdop')

image_filenames = data_list_files('figure4', 'Trial_7_no_subdop')

# Filter filenames
image_filenames = [f for f in image_filenames if f.endswith('ms.tiff')]
tof_dict_no_subdop = plot_calc_temp_full(base_path, image_filenames, plotting=False, save_result=False)


### II: PZT freq ramp only

In [ ]:
base_base_path = data_join('figure4')
base_path = os.path.join(base_base_path, 'Trial_8_freq_ramp_only')

image_filenames = data_list_files('figure4', 'Trial_8_freq_ramp_only')

# Filter filenames
image_filenames = [f for f in image_filenames if f.endswith('ms.tiff')]
tof_dict_PZT_freq_ramp_only = plot_calc_temp_full(base_path, image_filenames, plotting=False, save_result=False)


### III: PZT freq ramp + AOM power ramp

In [ ]:
base_base_path = data_join('figure4')
base_path = os.path.join(base_base_path, 'Trial_9_both_ramps')

image_filenames = data_list_files('figure4', 'Trial_9_both_ramps')

# Filter filenames
image_filenames = [f for f in image_filenames if f.endswith('ms.tiff')]
tof_dict_PZT_freq_SOA_power = plot_calc_temp_full(base_path, image_filenames, plotting=False, save_result=False)


### IV (ref): AOM freq ramp, SOA power ramp

In [ ]:
base_base_path = data_join('figure4')
base_path = os.path.join(base_base_path, '072525_AOM_freq_ramp_SOA_pow_ramp')

image_filenames = data_list_files('figure4', '072525_AOM_freq_ramp_SOA_pow_ramp')

# Filter filenames
image_filenames = [f for f in image_filenames if f.endswith('ms.tiff')]
tof_dict_AOM_freq_SOA_power = plot_calc_temp_full(base_path, image_filenames, plotting=False, save_result=False)


### All together

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

dicts = [
    (tof_dict_no_subdop, "I"),
    (tof_dict_PZT_freq_ramp_only, "II"),
    (tof_dict_PZT_freq_SOA_power, "III"),
    (tof_dict_AOM_freq_SOA_power, "IV"),
]

colors = ['red', 'green', 'blue', 'gray'] 
markers = ['o', 's', '^', 'X']        

for i, (tdict, title) in enumerate(dicts):
    t2 = tdict['tof_vals']**2
    ry2 = tdict['ry2']
    rz2 = tdict['rz2']
    fittedy = tdict['fittedy']
    fittedz = tdict['fittedz']
    Tempy = tdict['Tempy']
    Terr_y = tdict['Terr_y']
    Tempz = tdict['Tempz']
    Terr_z = tdict['Terr_z']

    # Plot the average of fittedy and fittedz as a single dashed line
    avg_fit = 0.5 * (fittedy + fittedz)
    Temp_avg = 0.5 * (Tempy + Tempz)
    Temp_avg_error = 0.5 * (Terr_y + Terr_z)

    # Plot σ_y
    ax.plot(t2, ry2, markers[i], color=colors[i], label=rf"{title}: $\sigma_{{y,z}}$, $T_{{\rm avg}}$ = {Temp_avg:.0f} µK")
    ax.plot(t2, rz2, markers[i], color=colors[i], mfc='none')
    ax.plot(t2, avg_fit, color=colors[i], ls='--', lw=2)

ax.set_xlabel('$t^2_{TOF}$ (ms$^2$)', fontsize=16)
ax.set_ylabel(rf'Cloud size $\sigma^2$ (mm$^2$)', fontsize=16)
#ax.set_title("Time-of-flight for Each Sequence", fontsize=16)
ax.legend(fontsize=14, loc='best')
plt.tight_layout()
ax.set_xlim(0, 37)
ax.set_ylim(-0.01, 0.45)
plt.show()

## Figure 5

In [ ]:
def atom_number_calc(C_tots, I_tot, detuning, t_exp):
    # C_tot is the total (background subtracked) counts of the pixel values in MOT region of interest
    # I_tot is total brightness of illumination beams in mW/cm^2
    # Detuning (in MHz) is the cooling beam detuning from F'=3 transition
    # t_exp is the total camera exposure time in milliseconds
    
    detuning_ang = 2*np.pi*detuning
    
    Gamma = 2*np.pi*6.066 #(Rb-87 natural linewidth for the D2 5^2𝑆_1/2 → 5^2𝑃_3/2 transition, in angular MHz)
    I_sat = 3.577 #Rb-87 saturation intensity (in mW/cm^2) for that same transition
    
    G = 10.42 #Gain coefficient for camera sensor (in e-/LSB, least significant bit)
    eta_QM = 0.45 #quantum efficiency of camera at 780 nm (in e-/photon)
    
    eta_geo = 0.74 #portion of light collected to to an obstruction in imaging path
    Omega = 0.035 #Solid angle of light collection (in sterradians)
    
    R_scat = Gamma/2.0*((I_tot/I_sat)/(1+(I_tot/I_sat)+4*(detuning_ang/Gamma)**2))
    
    dN_gamma_dt = (4*np.pi*C_tots*G)/(Omega*eta_QM*eta_geo*t_exp)*10**(-3)
    
    Ns = dN_gamma_dt/R_scat
    
    return Ns


### Fig. 5b

#### Rb ref

In [ ]:
base_path = data_join('figure5', 'fig5bcd')
filename = 'BN_Rb_ref_DataLog-LONG_gate1.2ms.csv'
path = os.path.join(base_path, filename)
freq_data = np.genfromtxt(path, delimiter=",")
gatetime = 1.2e-3
fs = 1/gatetime
times = np.arange(len(freq_data))*gatetime

bn_times_Rb_ref = times
bn_freq_data_Rb_ref = freq_data

# convert MOT brightness to atom number
# with sat. spec. re-referencing
filename = 'Rb_ref_data_with_sat_spec.csv'
path = os.path.join(base_path, filename)
data_ss = pd.read_csv(path)
data_np_ss = data_ss.to_numpy()

times_ss = data_np_ss[:,0]
counts_per_px_ss = data_np_ss[:,1]

# specific to individual data collection
ROI_tot_px = 150*150
exp = 0.5*10 #10 frame accumulation for this data
C_tots_ss = ROI_tot_px*counts_per_px_ss
Ns_ss = atom_number_calc(C_tots_ss, 59.0/2, 25.5, exp)
mot_atomN_time_Rb_ref = times_ss
mot_atomN_brightness_Rb_ref = Ns_ss

plt.plot(times/60, freq_data/1e6)
plt.xlabel("Time (in min)")
plt.ylabel('$\Delta$f (MHz)')
plt.title("Sub-doppler sequence \n ref. laser locked at 1'-3' crossover")
plt.show()

plt.figure(figsize=(4.5,3.5))
plt.plot(times+0.05, freq_data/1e6, lw=1.5)
plt.xlabel("Time (sec)", fontsize=20)
plt.ylabel('$\Delta$f (MHz)', fontsize=20)
plt.xlim([1,2])
plt.show()


t_start = 0.075
period = 0.304028
period_mask = (times >= t_start) & (times < t_start + period)
times_period = times[period_mask]
freq_data_period = freq_data[period_mask]

# Calculate the number of complete periods we can extract from the data
end_time = times[-1]
n_periods = int((end_time - t_start) // period)
n_plot_periods = min(100, n_periods)  # Only plot the first 10 periods, or less if not enough data

plt.figure(figsize=(10,2))

def plot_period(ax, idx, label=None, color=None):
    segment_start = t_start + idx*period
    segment_end = segment_start + period
    segment_mask = (times >= segment_start) & (times < segment_end)
    times_segment = times[segment_mask] - segment_start  # reset time to 0 for each period
    freq_segment = freq_data[segment_mask] / 1e6  # MHz
    if len(times_segment) > 0:
        ax.plot(times_segment*1e3, freq_segment, ls='-', lw=1, alpha=0.8, label=label, color=color)

ax = plt.gca()

# Plot just the first 100 periods
for i in range(n_plot_periods):
    plot_period(ax, i, color='k')
plt.xlabel("Time (ms)", fontsize=20)
plt.ylabel("$\Delta f$ (MHz)", fontsize=20)
plt.xlim([0, 310])
plt.show()


#### Not Rb referenced

In [ ]:
base_path = data_join('figure5', 'fig5bcd')
filename = 'BN_no_Rb_ref_DataLog-LONG-NOREREF_gate1.2ms.csv'
path = os.path.join(base_path, filename)
freq_data = np.genfromtxt(path, delimiter=",")
gatetime = 1.2e-3
fs = 1/gatetime
times = np.arange(len(freq_data))*gatetime

bn_times_no_Rb_ref = times
bn_freq_data_no_Rb_ref = freq_data

# convert MOT brightness to atom number
filename = 'no_Rb_ref_data_without_sat_spec.csv'
path = os.path.join(base_path, filename)
data_ss = pd.read_csv(path)
data_np_ss = data_ss.to_numpy()

times_ss = data_np_ss[:,0]
counts_per_px_ss = data_np_ss[:,1]

# specific to individual data collection
ROI_tot_px = 150*150
exp = 0.5*10 #10 frame accumulation for this data

C_tots_ss = ROI_tot_px*counts_per_px_ss

Ns_ss = atom_number_calc(C_tots_ss, 59.0/2, 25.5, exp)

plt.figure()
plt.plot(times_ss,Ns_ss,'*')
plt.title('Atom number v. time (with sat. spec. re-referencing)')
plt.xlabel('time (s)')
plt.ylabel('atom number')
plt.ylim(0, max(Ns_ss)*1.1)
plt.show()

mot_atomN_time_no_Rb_ref = times_ss
mot_atomN_brightness_no_Rb_ref = Ns_ss


In [ ]:

tmax = 10

# ---------- Split Y-axis setup ----------
fig, (ax1_top, ax1_bot) = plt.subplots(
    2, 1, sharex=True,
    figsize=(9, 4.5),
    gridspec_kw={'height_ratios': [1, 1], 'hspace': 0.06}
)

# Right y-axis (one per subplot so the red trace stays continuous)
ax2_top = ax1_top.twinx()
ax2_bot = ax1_bot.twinx()

# ---------- Top section ----------
ax1_top.plot(bn_times_Rb_ref/60, bn_freq_data_Rb_ref/1e6,color='tab:blue', label='Rb ref')
ax1_top.axhline(78, ls='--', color='k', lw=1.5)
ax1_top.set_ylim(70, 200)
ax1_top.set_xlim(0, tmax)

# ---------- Bottom section ----------
ax1_bot.plot(bn_times_no_Rb_ref/60, bn_freq_data_no_Rb_ref/1e6, color='tab:blue', lw=1, label='No Rb ref')
ax1_bot.axhline(np.min(bn_freq_data_no_Rb_ref/1e6),ls='--', lw=1.5, color='k')
ax1_bot.set_ylim(70, 250)
ax1_bot.set_xlim(0, tmax)

# ---------- Right axis: MOT signal ----------
ax2_top.plot(mot_atomN_time_Rb_ref/60, mot_atomN_brightness_Rb_ref/1e6, color='tab:red', lw=2)#, label='Mean MOT brightness')
ax2_top.tick_params(axis='y', labelcolor='tab:red')
ax2_top.set_ylim(-0.1, 2.1)
ax2_top.set_xlim(0, tmax)

ax2_bot.plot(mot_atomN_time_no_Rb_ref/60, mot_atomN_brightness_no_Rb_ref/1e6, color='tab:red', lw=2)#, label='Mean MOT brightness')
ax2_bot.tick_params(axis='y', labelcolor='tab:red')
ax2_bot.set_ylim(-0.1, 2.1)
ax2_bot.set_xlim(0, tmax)

fig.supxlabel("Time (min)", fontsize=20, y=-0.02)
fig.supylabel("Beat-note $\Delta f$ (MHz)", x=0.03, fontsize=20, color='tab:blue')

fig.text(0.96, 0.5, r"MOT $N_{atoms}$ ($\times 10^6$)", va='center', rotation=90,
         color='tab:red', fontsize=20)

for ax in (ax1_top, ax1_bot):
    ax.tick_params(axis='y', labelcolor='tab:blue')

ax1_top.tick_params(labelbottom=False)

# ---------- Legend (combine both axes’ handles) ----------
lines1, labels1 = ax1_top.get_legend_handles_labels()
lines2, labels2 = ax1_bot.get_legend_handles_labels()

plt.show()

### Fig. 5e

In [ ]:
base_path = data_join('figure5', 'fig5e')
filename = 'DataLogBUCKy-10US.0_longer_Rbref.csv'
path = os.path.join(base_path, filename)
freq_data1 = np.genfromtxt(path, delimiter=",")
gatetime = 10/0.8*2e-6
fs = 1/gatetime
times1 = np.arange(len(freq_data1)) * gatetime

# Select just one cycle in the times [19.6, 24.55]
cycle_mask1 = (times1 >= 19.6) & (times1 <= 24.55)
cycle_times1 = times1[cycle_mask1]
cycle_freq_data1 = freq_data1[cycle_mask1]

plt.figure(figsize=(4.75,4.5))
plt.plot(cycle_times1 - cycle_times1[0], cycle_freq_data1/1e6, marker='.', markersize=0.075, ls='', label='Rb ref',  color='k')
plt.ylim([150, 425])
plt.xlabel("Time (sec)", fontsize=20)
plt.ylabel(r'$\Delta$f (MHz)', fontsize=20)
plt.tight_layout()
plt.show()
